In [63]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from scipy.spatial import ConvexHull
import plotly.express as px
import os

In [64]:
def plot_confidence_ellipse(data, ax, n_std=2, **kwargs):
    """   
    Plot a confidence ellipse for 2D data.
    """
    cov = np.cov(data, rowvar=False)  # Covariance matrix
    mean = np.mean(data, axis=0)     # Mean of the data

    # Compute the eigenvalues and eigenvectors
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Compute the angle and width/height of the ellipse
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigvals)

    # Draw the ellipse
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)
    return ellipse

English note: release cleanup annotation.


In [65]:
projectpath = '/mnt/chromeos/MyFiles/Downloads/XJN_M2project'
test = 'test1'##VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5
match test:
    case 'test1':
        miceID = '1'#VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5
        cell_type = 'CamkII_TST' 
        signal_path = os.path.join(projectpath, 'signal_data',test, cell_type, miceID, 'signal_save', f'not_aligned_{cell_type}{miceID}_TST.csv')
        im_path = os.path.join(projectpath, 'behavioral_data', test, cell_type, miceID, 'im_preprocessed.csv')
        out_path = os.path.join(projectpath, 'summary', test, cell_type, miceID, 'PCA_Analysis')
    case 'test2':
        ##############
        #mice_IDs = ["LHQ50","LHQ30","LH5799","LH5798","LH0166","LH0167","LH5779",
        #    "CSDS0087","CSDS0103","CSDS0126","CSDS0179","CSDS0370",
        #    "CSDSQ29", "CSDSQ27", "CSDSQ26", "CSDS5797", "CSDS5776"]
        ###############
        miceID = 'LHQ30'
        condition = 'rescue' #pre, post
        experiment = 'TST'
        align = 'aligned'
        signal_path = os.path.join(projectpath, 'signal_data', test, condition, miceID,'signal_save',f'{align}_{miceID}_{experiment}.csv')
        im_path = os.path.join(projectpath, 'behavioral_data', test, condition, miceID, 'im_preprocessed.csv')
        out_path = os.path.join(projectpath, 'summary', test, condition, miceID,experiment, 'PCA_Analysis')
if not os.path.exists(out_path):os.makedirs(out_path)
data = pd.read_csv(signal_path)
im_data = pd.read_csv(im_path)

In [ ]:
data.shape

In [67]:
maxlen = min(data.shape[0],im_data.shape[0])
if data.shape[0]>maxlen:
    data = data.iloc[:maxlen,:]
if im_data.shape[0]>maxlen:
    im_data = im_data.iloc[:maxlen,:]

In [ ]:
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data.to_numpy())
explained_variance = pca.explained_variance_ratio_
explained_variance

In [ ]:
data_pca.shape

In [70]:
time_axis = np.arange(data_pca.shape[0]).reshape(-1, 1)
data_3d = np.hstack((data_pca, time_axis))
data_3d = data_3d.astype(np.float64)
data_3d = pd.DataFrame(data_3d, columns=['PCA1', 'PCA2', 'time'])
data_4d = pd.concat([data_3d, im_data], axis=1)

match test:
    case 'test1':
        data_4d.to_csv(os.path.join(out_path, f'{cell_type}{miceID}_PCAComponents.csv'), index=False)
    case 'test2':
        data_4d.to_csv(os.path.join(out_path, f'{miceID}{condition}_PCAComponents.csv'), index=False)

In [ ]:
fig = px.scatter_3d(data_4d, x='PCA1', y='PCA2', z='time',
                    color='processed_state',
                    title='3D PCA Motion'
                    )
fig.show()
plt.close('all')

In [ ]:


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
color_map = {0: 'blue', 1: 'red'}
# Group by the label column ('processed_state') and plot each group
for label, group in data_4d.groupby('processed_state'):
    ax.scatter(group['PCA1'], group['PCA2'], group['time'], label=f'Label {label}', alpha=0.5, c=color_map[label])

# Add axis labels and title
ax.set_xlabel('PCA1')
ax.set_ylabel('PCA2')
ax.set_zlabel('Time')
ax.set_zlim([0,4000])
ax.set_title('3D PCA Motion with Labels')
ax.view_init(azim=65)

# Add legend
ax.legend(title='Processed State')

# Show the plot
match test:
    case 'test1':
       plt.savefig(os.path.join(out_path, f'{cell_type}{miceID}_3D_PCAMotion.pdf'), dpi=300)
    case 'test2':
       plt.savefig(os.path.join(out_path, f'{miceID}{condition}{experiment}_3D_PCAMotion.pdf'), dpi=300)
plt.show()
plt.close('all')


In [ ]:
# Create a 2D scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

# Define a color map for binary labels (0 -> blue, 1 -> red)
# Group by the label column ('processed_state') and plot each group
for label, group in data_4d.groupby('processed_state'):
    ax.scatter(
        group['PCA1'], 
        group['PCA2'], 
        label=f'Label {label}', 
        alpha=0.7, 
        c=color_map[label]
    )
    #plot_confidence_ellipse(group[['PCA1', 'PCA2']].to_numpy(), ax, n_std=2.5, edgecolor=color_map[label][0], facecolor='none', linewidth=2)

# Add axis labels and title
ax.set_xlabel('PCA1 ')
ax.set_ylabel('PCA2 ')
ax.set_title('2D PCA Motion')

# Add legend
ax.legend(title='Processed State')

# Save the plot
match test:
    case 'test1':
        plt.savefig(out_path + '/' + cell_type + miceID + '_2D_PCAMotion.pdf', dpi=300)

# Show the plot